# Bronze data-domain profile

Profiles low-cardinality string columns that are defined in the schema contract. Each run replaces the stored domain for the profiled Bronze table and column; domains with more than 40 values are deliberately not retained.

In [ ]:
BRONZE_SCHEMA = "bronze"
CONTRACT_TABLE = "monitoring.cfg_schema_contract_column"
DOMAIN_TABLE = "monitoring.cfg_data_domain"
MAX_DISTINCT_VALUES = 40
INCLUDE_EMPTY_STRING = False

# This standalone control notebook owns its configuration bootstrap.
CFG_NOTEBOOK_NAME = "00_setup_cfg"
AUDIT_TABLE = "monitoring.cfg_silver_export_load"
TIME_PARSER_POLICY = "CORRECTED"
NOTEBOOK_TIMEOUT_SECONDS = 1800
JOB_RUN_ID = ""


In [ ]:
from notebookutils import mssparkutils

cfg_result = mssparkutils.notebook.run(
    CFG_NOTEBOOK_NAME,
    NOTEBOOK_TIMEOUT_SECONDS,
    {
        "AUDIT_TABLE": AUDIT_TABLE,
        "TIME_PARSER_POLICY": TIME_PARSER_POLICY,
        "JOB_RUN_ID": JOB_RUN_ID,
    },
)
print(f"Configuration setup completed: {cfg_result}")

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

RUN_ID = str(uuid4())
PROFILED_AT = datetime.now(timezone.utc)

DOMAIN_SCHEMA = StructType([
    StructField("source_schema", StringType(), False),
    StructField("source_table", StringType(), False),
    StructField("column_name", StringType(), False),
    StructField("contract_data_type", StringType(), True),
    StructField("data_domain", StringType(), False),
    StructField("distinct_value_count", LongType(), False),
    StructField("profiled_at", TimestampType(), False),
    StructField("run_id", StringType(), False),
    StructField("job_run_id", StringType(), True),
])


def normalise_name(value):
    return "".join(character for character in str(value).lower() if character.isalnum())


def quote_identifier(identifier):
    return "`" + identifier.replace("`", "``") + "`"


def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"


def is_contract_string(data_type):
    """Return whether a PostgreSQL contract type is represented as a Spark string."""
    type_name = (data_type or "").strip().lower()
    return any(marker in type_name for marker in ("character", "varchar", "text", "uuid", "json", "string"))


def domain_predicate(source_table, column_name):
    return (
        f"source_schema = {sql_literal(BRONZE_SCHEMA)} AND "
        f"source_table = {sql_literal(source_table)} AND "
        f"column_name = {sql_literal(column_name)}"
    )


def replace_domain(source_table, column_name, contract_data_type, values):
    """Atomically replace a column domain, or remove a stale one when it is not eligible."""
    predicate = domain_predicate(source_table, column_name)
    if not values:
        DeltaTable.forName(spark, DOMAIN_TABLE).delete(predicate)
        return

    row_count = len(values)
    rows = [
        (
            BRONZE_SCHEMA, source_table, column_name, contract_data_type, value,
            row_count, PROFILED_AT, RUN_ID, JOB_RUN_ID or None,
        )
        for value in values
    ]
    (
        spark.createDataFrame(rows, DOMAIN_SCHEMA)
        .write.format("delta")
        .mode("overwrite")
        .option("replaceWhere", predicate)
        .saveAsTable(DOMAIN_TABLE)
    )


In [ ]:
if not spark.catalog.tableExists(CONTRACT_TABLE):
    raise RuntimeError(f"Schema contract table does not exist: {CONTRACT_TABLE}")
if not spark.catalog.tableExists(DOMAIN_TABLE):
    raise RuntimeError(f"Data-domain table does not exist: {DOMAIN_TABLE}")

# Build a case- and punctuation-insensitive map from the contract. Conflicting
# entries are not profiled: the contract must be unambiguous before it governs a domain.
contract_definitions = {}
ambiguous_contract_columns = set()
for row in spark.table(CONTRACT_TABLE).select("table_name", "column_name", "data_type").collect():
    if not row.table_name or not row.column_name or not is_contract_string(row.data_type):
        continue
    key = (normalise_name(row.table_name), normalise_name(row.column_name))
    existing_type = contract_definitions.get(key)
    if existing_type is not None and existing_type != row.data_type:
        ambiguous_contract_columns.add(key)
    else:
        contract_definitions[key] = row.data_type
for key in ambiguous_contract_columns:
    contract_definitions.pop(key, None)

bronze_tables = {}
for row in spark.sql(f"SHOW TABLES IN {quote_identifier(BRONZE_SCHEMA)}").collect():
    if not row.isTemporary:
        bronze_tables.setdefault(normalise_name(row.tableName), row.tableName)

profiled_columns = 0
skipped_high_cardinality = 0
skipped_empty = 0
missing_bronze_columns = 0

for (logical_table, logical_column), contract_data_type in sorted(contract_definitions.items()):
    source_table = bronze_tables.get(logical_table)
    if source_table is None:
        missing_bronze_columns += 1
        continue

    source_frame = spark.table(f"{quote_identifier(BRONZE_SCHEMA)}.{quote_identifier(source_table)}")
    actual_columns = {normalise_name(field.name): field.name for field in source_frame.schema.fields}
    source_column = actual_columns.get(logical_column)
    if source_column is None:
        missing_bronze_columns += 1
        continue

    string_value = F.col(quote_identifier(source_column)).cast("string")
    domain_values = source_frame.select(string_value.alias("data_domain")).where(F.col("data_domain").isNotNull())
    if not INCLUDE_EMPTY_STRING:
        domain_values = domain_values.where(F.length(F.trim(F.col("data_domain"))) > 0)

    # Collect at most one value beyond the permitted limit, so high-cardinality
    # domains are never written and their full set is never brought to the driver.
    values = [
        domain_row.data_domain
        for domain_row in domain_values.distinct().limit(MAX_DISTINCT_VALUES + 1).collect()
    ]
    if len(values) > MAX_DISTINCT_VALUES:
        replace_domain(source_table, source_column, contract_data_type, [])
        skipped_high_cardinality += 1
        continue
    if not values:
        replace_domain(source_table, source_column, contract_data_type, [])
        skipped_empty += 1
        continue

    values.sort()
    replace_domain(source_table, source_column, contract_data_type, values)
    profiled_columns += 1

summary = {
    "run_id": RUN_ID,
    "profiled_columns": profiled_columns,
    "skipped_high_cardinality": skipped_high_cardinality,
    "skipped_empty": skipped_empty,
    "missing_bronze_columns": missing_bronze_columns,
    "ambiguous_contract_columns": len(ambiguous_contract_columns),
}
print(summary)